### TAREA 1.1 DATASET INCENDIOS MITECO

#### ANÁLISIS EXPLORATORIO INICIAL DEL DATASET DE INDENCIOS

In [1]:
import pandas as pd


df = pd.read_excel('datos_incendios.xlsx')


print("--- COLUMNAS DEL DATASET ---")
print(df.columns.tolist())


print("\n--- INFORMACIÓN GENERAL ---")
print(df.info())


print("\n--- MUESTRA DE DATOS ---")
display(df.head()) 



if 'PROVINCIA' in df.columns:
    print("\n--- VALORES ÚNICOS EN PROVINCIA ---")
    print(df['PROVINCIA'].unique())

--- COLUMNAS DEL DATASET ---
['Campania', 'NumeroParte', 'Estado', 'Comunidad', 'Provincia', 'Municipio', 'ComarcaIsla', 'EntidadMenor', 'NumeroMunicipiosAfectados', 'Hoja', 'Cuadricula', 'Huso', 'CoordenadaX', 'CoordenadaY', 'Datum', 'NumeroPuntosInicioIncendio', 'Detectado', 'Extinguido', 'Causa', 'Motivacion', 'SuperficieArbolada', 'SuperficieNoArbolada', 'SuperficieTotalForestal', 'SuperficieAgricola', 'OtrasSuperficiesNoforestales', 'AfectoZonasInterfazUrbanoForestal', 'TipoInterfazAfectado', 'AfectoEspacioProtegido', 'AfectoTierrasAgrarias', 'AfectoZar', 'NumeroPartePss']

--- INFORMACIÓN GENERAL ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4498 entries, 0 to 4497
Data columns (total 31 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   Campania                           4498 non-null   int64 
 1   NumeroParte                        4498 non-null   int64 
 2   Estado                

,Campania,NumeroParte,Estado,Comunidad,Provincia,Municipio,ComarcaIsla,EntidadMenor,NumeroMunicipiosAfectados,Hoja,...,SuperficieNoArbolada,SuperficieTotalForestal,SuperficieAgricola,OtrasSuperficiesNoforestales,AfectoZonasInterfazUrbanoForestal,TipoInterfazAfectado,AfectoEspacioProtegido,AfectoTierrasAgrarias,AfectoZar,NumeroPartePss
0,2010,2010030001,Migración,C. VALENCIANA,ALICANTE,AIGÜES,L ALACANTI,,1,805,...,"13,0000","15,0000","0,0000","0,0000",Sin determinar,,No,No,No,
1,2010,2010030002,Migración,C. VALENCIANA,ALICANTE,MILLENA,EL COMTAT,,1,804,...,"0,2000","0,3000","0,2000","0,0000",Sin determinar,,No,No,No,
2,2010,2010030003,Migración,C. VALENCIANA,ALICANTE,GUARDAMAR DEL SEGURA,EL BAIX SEGURA (LA VEGA BAJA),,1,705,...,"0,0000","0,0100","0,0000","0,0000",Sin determinar,,No,No,No,
3,2010,2010030004,Migración,C. VALENCIANA,ALICANTE,BENIFATO,LA MARINA BAIXA,,1,804,...,"0,7000","0,7500","0,0000","0,0000",Sin determinar,,No,No,No,
4,2010,2010030005,Migración,C. VALENCIANA,ALICANTE,BENIDORM,LA MARINA BAIXA,,1,805,...,"0,0000","0,0200","0,0000","0,0000",Sin determinar,,No,No,No,


#### Limpiez e Ingeniería de Variables

In [2]:
import pandas as pd
from pyproj import Transformer
import numpy as np



def limpiar_numero(valor):
    """Convierte strings con comas (decimales europeos) a floats de Python."""
    if pd.isna(valor): 
        return 0.0
    if isinstance(valor, str):
        # Eliminamos espacios y cambiamos coma por punto
        valor_limpio = valor.strip().replace(',', '.')
        try:
            return float(valor_limpio)
        except ValueError:
            return 0.0
    return float(valor)

# Creamos los transformadores fuera de la función para mayor eficiencia
# EPSG:25830 (Huso 30) y EPSG:25831 (Huso 31) -> WGS84 (Lat/Lon)
trans30 = Transformer.from_crs("epsg:25830", "epsg:4326", always_xy=True)
trans31 = Transformer.from_crs("epsg:25831", "epsg:4326", always_xy=True)

def proyectar_coordenadas(fila):
    """Convierte UTM a Lat/Lon basándose en la columna 'Huso'."""
    x = fila['CoordenadaX_num']
    y = fila['CoordenadaY_num']
    huso = str(fila['Huso'])
    
    if x == 0 or y == 0:
        return None, None
    
    try:
        
        if '31' in huso:
            return trans31.transform(x, y)
        else:
            return trans30.transform(x, y)
    except:
        return None, None


cols_a_limpiar = [
    'CoordenadaX', 'CoordenadaY', 'SuperficieArbolada', 
    'SuperficieNoArbolada', 'SuperficieTotalForestal', 
    'SuperficieAgricola', 'OtrasSuperficiesNoforestales'
]

for col in cols_a_limpiar:
    
    df[f'{col}_num'] = df[col].apply(limpiar_numero)



print("Transformando coordenadas (UTM -> Lat/Lon)...")
df[['lon', 'lat']] = df.apply(proyectar_coordenadas, axis=1, result_type='expand')



print("Procesando fechas y duraciones...")

df['fecha_ini'] = pd.to_datetime(df['Detectado'], dayfirst=True, errors='coerce')
df['fecha_fin'] = pd.to_datetime(df['Extinguido'], dayfirst=True, errors='coerce')


df['duracion_h'] = (df['fecha_fin'] - df['fecha_ini']).dt.total_seconds() / 3600



# Calculamos la Superficie Total Real sumando todas las categorías
df['Superficie_Total_Real'] = (
    df['SuperficieArbolada_num'] + 
    df['SuperficieNoArbolada_num'] + 
    df['SuperficieAgricola_num'] + 
    df['OtrasSuperficiesNoforestales_num']
)



# Definimos las columnas que realmente nos interesan para el TFG
columnas_finales = [
    'Campania', 'fecha_ini', 'duracion_h', 'Provincia', 'Municipio', 
    'Causa', 'Superficie_Total_Real', 'SuperficieArbolada_num', 
    'lat', 'lon', 'AfectoZonasInterfazUrbanoForestal', 'AfectoEspacioProtegido'
]

df_tfg = df.dropna(subset=['lat', 'lon', 'fecha_ini']).copy()


df_tfg.loc[df_tfg['duracion_h'] < 0, 'duracion_h'] = 0


df_tfg = df_tfg[columnas_finales].rename(columns={'SuperficieArbolada_num': 'Superficie_Arbolada'})



print("\n" + "="*30)
print("¡PROCESAMIENTO COMPLETADO!")
print(f"Registros originales: {len(df)}")
print(f"Registros finales válidos: {len(df_tfg)}")
print("="*30)


df_tfg.to_csv('incendios_CV_limpio.csv', index=False, sep=';', encoding='utf-8-sig')
print("Archivo guardado como: incendios_CV_limpio.csv")
print(df_tfg.head())

Transformando coordenadas (UTM -> Lat/Lon)...
Procesando fechas y duraciones...

¡PROCESAMIENTO COMPLETADO!
Registros originales: 4498
Registros finales válidos: 4478
Archivo guardado como: incendios_CV_limpio.csv
   Campania           fecha_ini  duracion_h Provincia             Municipio  \
0      2010 2010-01-01 14:48:00   23.200000  ALICANTE                AIGÜES   
2      2010 2010-02-10 13:50:00    0.916667  ALICANTE  GUARDAMAR DEL SEGURA   
3      2010 2010-03-02 13:35:00    4.750000  ALICANTE              BENIFATO   
4      2010 2010-03-09 12:30:00    1.166667  ALICANTE              BENIDORM   
5      2010 2010-03-15 16:15:00    2.750000  ALICANTE       MONÒVER/MONÓVAR   

                                              Causa  Superficie_Total_Real  \
0                               [400]  Intencionado                  15.00   
2                               [400]  Intencionado                   0.01   
3       [284]  Limpieza de lindes y bordes de finca                   0.75   